# Reto: agentes inteligentes en un laberinto

Este notebook presenta un generador basado en Kruskal y compara un agente DFS determinista con un explorador estocástico por fronteras. La implementación se importa desde el paquete `laberinto` para mantener una sola fuente de verdad.

In [ ]:
from pathlib import Path
import sys

RAIZ = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from laberinto import AgenteDFS, AgenteExploradorFronteras, Entorno, crear_laberinto
from laberinto.visualizacion import dibujar_laberinto

## 1. Generación del entorno

El generador construye entre uno y tres componentes mediante Kruskal aleatorio. Después agrega hasta cuatro conexiones internas. No busca caminos ni comprueba si la salida es alcanzable.

In [ ]:
FILAS, COLUMNAS, SEMILLA = 15, 15, 26
laberinto, entrada, salida = crear_laberinto(FILAS, COLUMNAS, SEMILLA)
print(f'Entrada: {entrada} | Salida: {salida}')
dibujar_laberinto(laberinto, entrada, salida)

## 2. DFS determinista con poda

El agente conserva el avance y retroceso de DFS, pero poda ciclos, rutas que llegan a una celda con igual o mayor profundidad y ramas incapaces de mejorar la mejor solución.

In [ ]:
dfs = AgenteDFS(Entorno(laberinto))
informe_dfs, eventos_dfs = dfs.ejecutar()
informe_dfs

In [ ]:
evento_final_dfs = eventos_dfs[-1]
dibujar_laberinto(laberinto, entrada, salida, evento_final_dfs, solo_ruta=True)

## 3. Exploración estocástica por fronteras

El agente solo percibe su celda actual. A* no intenta adivinar la salida: calcula rutas hacia fronteras ya descubiertas dentro del mapa conocido.

In [ ]:
explorador = AgenteExploradorFronteras(Entorno(laberinto), semilla=SEMILLA)
informe_fronteras, eventos_fronteras = explorador.ejecutar()
informe_fronteras

In [ ]:
evento_final = eventos_fronteras[-1]
dibujar_laberinto(laberinto, entrada, salida, evento_final, vista_agente=True)

## 4. Comparación

DFS con poda conserva el comportamiento de backtracking, reduce recorridos redundantes y confirma el mejor camino. La exploración por fronteras representa mejor un agente situado: construye memoria, se desplaza físicamente y desconoce la salida. Su trayectoria no tiene por qué ser la ruta más corta, pero la ruta final siempre utiliza conexiones descubiertas.

In [ ]:
comparacion = {
    'DFS determinista': {
        'solucion': informe_dfs['tiene_solucion'],
        'expandidas': informe_dfs['expandidas'],
        'podas': informe_dfs['podas'],
        'longitud_ruta': len(informe_dfs['camino'] or []),
    },
    'Fronteras': {
        'solucion': informe_fronteras['tiene_solucion'],
        'expandidas': informe_fronteras['expandidas'],
        'movimientos': informe_fronteras['movimientos'],
        'longitud_ruta': len(informe_fronteras['camino'] or []),
    },
}
comparacion